# Experiment Plots

This notebook renders experiment figures from saved CSV/JSON artifacts only. It does not run raw experiment analysis.

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns

FONT_SCALE = 1.0
TITLE_SIZE = 30
LABEL_SIZE = 24
TICK_SIZE = 21
LEGEND_SIZE = 21
ANNOTATION_SIZE = 22

sns.set_theme(style="whitegrid", context="paper", font_scale=FONT_SCALE)
plt.rcParams.update({
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "figure.titlesize": TITLE_SIZE,
})

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError(f"Could not find repository root from {start}")

ROOT = find_repo_root(Path.cwd())
SAVE_PLOTS = True

EXP1_DIR = ROOT / "experiments/exp1_relevance_bridge/results"
EXP2_DIR = ROOT / "experiments/exp2_iceberg/results"
EXP4_DIR = ROOT / "experiments/exp4_implicature_flow/results"
EXP4_SANKEY_DIR = ROOT / "experiments/exp4_implicature_flow/episode_sankey_html"
EXP5_RESULT_DIRS: tuple[Path, ...] = (
    ROOT / "experiments/exp5_processing_load/results/sentence-transformers__all-MiniLM-L6-v2",
    ROOT / "experiments/exp5_processing_load/results/Qwen__Qwen3-Embedding-4B",
)
EXP6_DIR = ROOT / "experiments/exp6_quantity_repair_cascades/results"
EXP7_DIR = ROOT / "experiments/exp7_social_power/results"

EXP2_MIN_PLOT_TRANSITION_ROWS = 1000
EXP6_SAMPLE_CASCADE_COUNT = 12
EXP6_TRIGGER_ORDER = ("under_info", "over_info")
EXP6_TRIGGER_LABELS = {"under_info": "Under-info", "over_info": "Over-info"}
EXP6_TRIGGER_COLORS = {"under_info": "#0f4c5c", "over_info": "#bc6c25"}
EXP6_EVENT_MARKERS = {"repair_event": "o", "challenge_event": "s", "continuation_event": "^", "support_marker": "x", "exit_marker": "D"}
EXP6_EVENT_COLORS = {"repair_event": "#1f77b4", "challenge_event": "#d62728", "continuation_event": "#6a994e", "support_marker": "#7f7f7f", "exit_marker": "#2a9d8f"}
EXP7_ROLE_ORDER = ("guest", "host")
EXP7_VIOLATION_ORDER = ("Quantity", "Relation", "Manner")
EXP7_VIOLATION_SEVERITY = {"Quantity": 1, "Relation": 2, "Manner": 3}

def require_file(path: Path) -> Path:
    if not path.exists():
        raise FileNotFoundError(f"Required plot input is missing: {path}")
    return path

def read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(require_file(path))

def read_json(path: Path) -> dict[str, Any]:
    with require_file(path).open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {path}")
    return payload

def read_exp4_episode_metrics(report: dict[str, Any], path: Path) -> pd.DataFrame:
    if path.exists():
        return read_csv(path)
    rows = report.get("per_episode_metrics")
    if not isinstance(rows, list) or not rows:
        raise ValueError("Exp4 global report does not contain per_episode_metrics for backfilling exp4_episode_metrics.csv.")
    df = pd.DataFrame(rows)
    if SAVE_PLOTS:
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=False)
    return df

def read_exp4_plot_data(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        return read_csv(path)
    raise FileNotFoundError(
        f"Missing {label}: {path}. Run `python experiments/exp4_implicature_flow/exp4_implicature.py` "
        "once to generate Exp4 plot-data CSVs from the existing analysis code."
    )

def require_columns(df: pd.DataFrame, columns: Sequence[str], label: str) -> None:
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")

def pdf_path(path: Path) -> Path:
    return path.with_suffix(".pdf")

def apply_matplotlib_font_sizes(fig: plt.Figure) -> None:
    for axis in fig.axes:
        axis.title.set_fontsize(TITLE_SIZE)
        axis.xaxis.label.set_fontsize(LABEL_SIZE)
        axis.yaxis.label.set_fontsize(LABEL_SIZE)
        axis.tick_params(axis="both", labelsize=TICK_SIZE)
        legend = axis.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontsize(LEGEND_SIZE)
        for text in axis.texts:
            text.set_fontsize(ANNOTATION_SIZE)
    for legend in fig.legends:
        for text in legend.get_texts():
            text.set_fontsize(LEGEND_SIZE)

def save_figure(fig: plt.Figure, path: Path) -> None:
    if SAVE_PLOTS:
        output_path = pdf_path(path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        apply_matplotlib_font_sizes(fig)
        fig.tight_layout(pad=2.0)
        fig.savefig(output_path, format="pdf", bbox_inches="tight", pad_inches=0.30)

def save_plotly_pdf(fig: go.Figure, path: Path) -> None:
    if SAVE_PLOTS:
        output_path = pdf_path(path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.update_layout(font={"size": TICK_SIZE})
        try:
            fig.write_image(output_path, format="pdf")
        except ValueError as error:
            raise RuntimeError("Plotly PDF export requires the kaleido package from pyproject.toml.") from error

def robust_xlim(values: Sequence[float], lo: float, hi: float, cap: float | None) -> tuple[float, float]:
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if len(finite) == 0:
        raise ValueError("Cannot compute limits for an empty finite array.")
    xmin = float(np.quantile(finite, lo))
    xmax = float(np.quantile(finite, hi))
    if cap is not None:
        xmax = min(float(cap), xmax)
    if xmin == xmax:
        xmax = xmin + 1.0
    return xmin, xmax

def finite_numeric(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return values[np.isfinite(values)]


## Exp 1: Relevance Bridge

In [ ]:
def ci_errorbar_values(df: pd.DataFrame, mean_column: str, low_column: str, high_column: str, label: str) -> np.ndarray:
    means = df[mean_column].astype(float).to_numpy()
    lows = df[low_column].astype(float).to_numpy()
    highs = df[high_column].astype(float).to_numpy()
    errors = np.vstack([means - lows, highs - means])
    if not np.all(np.isfinite(errors)):
        raise ValueError(f"{label} has non-finite confidence interval values.")
    if np.any(errors < -1e-9):
        raise ValueError(f"{label} has confidence interval bounds that do not contain the mean.")
    return np.maximum(errors, 0.0)

category_summary = read_csv(EXP1_DIR / "exp1_llm_next_turn_by_category.csv")
require_columns(category_summary, ["category", "pair_count", "mean_rank_lift", "mean_rank_lift_ci95_low", "mean_rank_lift_ci95_high"], "Exp1 category data")

category_plot = category_summary.sort_values("mean_rank_lift", ascending=True).reset_index(drop=True)
if category_plot.empty:
    raise ValueError("Exp1 category data has no rows to plot.")

rank_positions = np.arange(len(category_plot), dtype=float)
rank_values = category_plot["mean_rank_lift"].astype(float).to_numpy()
rank_lows = category_plot["mean_rank_lift_ci95_low"].astype(float).to_numpy()
rank_highs = category_plot["mean_rank_lift_ci95_high"].astype(float).to_numpy()
rank_labels = [
    f"{category} (n={pair_count:,})"
    for category, pair_count in zip(category_plot["category"].astype(str), category_plot["pair_count"].astype(int), strict=True)
]
fig, ax = plt.subplots(figsize=(11.8, 7.0))
ax.errorbar(rank_values, rank_positions, xerr=ci_errorbar_values(category_plot, "mean_rank_lift", "mean_rank_lift_ci95_low", "mean_rank_lift_ci95_high", "Exp1 category rank lift"), fmt="none", ecolor="#6c757d", elinewidth=1.2, capsize=3)
ax.scatter(rank_values, rank_positions, color=np.where(rank_values >= 0.0, "#1b9e77", "#d95f02"), s=58, zorder=3)
ax.axvline(0.0, color="black", linestyle="--", linewidth=1.0)
rank_x_min = min(0.0, float(np.min(rank_lows)))
rank_x_max = max(0.0, float(np.max(rank_highs)))
rank_x_padding = max(0.02, 0.12 * (rank_x_max - rank_x_min))
ax.set_xlim(rank_x_min - rank_x_padding, rank_x_max + rank_x_padding)
ax.set_yticks(rank_positions)
ax.set_yticklabels(rank_labels, fontsize=TICK_SIZE)
ax.tick_params(axis="x", labelsize=TICK_SIZE)
ax.set_xlabel("Mean rank lift\n(positive = assumptions improve rank)", fontsize=LABEL_SIZE)
ax.set_ylabel("Category", fontsize=LABEL_SIZE)
fig.tight_layout()
save_figure(fig, EXP1_DIR / "exp1_political_llm_rank_lift_by_category.pdf")
plt.show()


## Exp 2: Iceberg

In [ ]:
bins_df = read_csv(EXP2_DIR / "exp2_local_effect_bins.csv")
heuristic_bins_df = read_csv(EXP2_DIR / "exp2_heuristic_local_effect_bins.csv")
require_columns(bins_df, ["delta_stance_pt", "n_transition_rows", "mean_delta_log_iceberg_density", "ci_low_delta_log_iceberg_density", "ci_high_delta_log_iceberg_density"], "Exp2 local effect bins")
require_columns(heuristic_bins_df, ["delta_stance_pt", "n_transition_rows", "mean_delta_log_words_per_second", "ci_low_delta_log_words_per_second", "ci_high_delta_log_words_per_second"], "Exp2 heuristic local effect bins")
plot_df = bins_df[bins_df["n_transition_rows"].astype(int) >= EXP2_MIN_PLOT_TRANSITION_ROWS].copy()
heuristic_plot_df = heuristic_bins_df[heuristic_bins_df["n_transition_rows"].astype(int) >= EXP2_MIN_PLOT_TRANSITION_ROWS].copy()
if plot_df.empty:
    raise ValueError(f"No Exp2 bins meet the minimum row threshold: {EXP2_MIN_PLOT_TRANSITION_ROWS}")
if heuristic_plot_df.empty:
    raise ValueError(f"No Exp2 heuristic bins meet the minimum row threshold: {EXP2_MIN_PLOT_TRANSITION_ROWS}")

comparison_df = plot_df.merge(heuristic_plot_df, on=["delta_stance_pt", "n_transition_rows"], how="inner")
if comparison_df.empty:
    raise ValueError("No shared Exp2 local-effect bins remain after aligning iceberg-density and word-rate summaries")

x_values = comparison_df["delta_stance_pt"].astype(float).to_numpy()
iceberg_values = comparison_df["mean_delta_log_iceberg_density"].astype(float).to_numpy()
iceberg_lower_values = comparison_df["ci_low_delta_log_iceberg_density"].astype(float).to_numpy()
iceberg_upper_values = comparison_df["ci_high_delta_log_iceberg_density"].astype(float).to_numpy()
word_rate_values = comparison_df["mean_delta_log_words_per_second"].astype(float).to_numpy()
word_rate_lower_values = comparison_df["ci_low_delta_log_words_per_second"].astype(float).to_numpy()
word_rate_upper_values = comparison_df["ci_high_delta_log_words_per_second"].astype(float).to_numpy()

fig, relationship_axis = plt.subplots(figsize=(10.6, 6.3))
relationship_axis.errorbar(x_values, iceberg_values, yerr=np.vstack([iceberg_values - iceberg_lower_values, iceberg_upper_values - iceberg_values]), fmt="o", color="#1f5c99", ecolor="#8fb8de", elinewidth=1.1, capsize=4, markersize=6, label="Directional stance")
relationship_axis.plot(x_values, iceberg_values, color="#1f5c99", linewidth=2.3)
relationship_axis.errorbar(x_values, word_rate_values, yerr=np.vstack([word_rate_values - word_rate_lower_values, word_rate_upper_values - word_rate_values]), fmt="s", color="#8d5a00", ecolor="#d9b77e", elinewidth=1.1, capsize=4, markersize=5, label="Word rate")
relationship_axis.plot(x_values, word_rate_values, color="#8d5a00", linewidth=2.3)
relationship_axis.axhline(0.0, color="#6c757d", linewidth=1.0, alpha=0.75)
relationship_axis.axvline(0.0, color="#6c757d", linewidth=1.0, alpha=0.30)
relationship_axis.set_xlabel("Change in stance")
relationship_axis.set_ylabel("Mean local change")
relationship_axis.legend(frameon=False, loc="best", fontsize=LEGEND_SIZE)
relationship_axis.grid(alpha=0.16, linewidth=0.6)
fig.tight_layout()
save_figure(fig, EXP2_DIR / "exp2_local_relationship.pdf")
if SAVE_PLOTS:
    fig.savefig(EXP2_DIR / "exp2_local_relationship.png", format="png", dpi=300, bbox_inches="tight", pad_inches=0.30)
plt.show()

heuristic_x_values = heuristic_plot_df["delta_stance_pt"].astype(float).to_numpy()
heuristic_y_values = heuristic_plot_df["mean_delta_log_words_per_second"].astype(float).to_numpy()
heuristic_lower_values = heuristic_plot_df["ci_low_delta_log_words_per_second"].astype(float).to_numpy()
heuristic_upper_values = heuristic_plot_df["ci_high_delta_log_words_per_second"].astype(float).to_numpy()

fig, ax = plt.subplots(figsize=(10.6, 6.3))
ax.errorbar(heuristic_x_values, heuristic_y_values, yerr=np.vstack([heuristic_y_values - heuristic_lower_values, heuristic_upper_values - heuristic_y_values]), fmt="o", color="#495057", ecolor="#adb5bd", elinewidth=1.2, capsize=4, markersize=6, label="Heuristic mean (95% CI)")
ax.plot(heuristic_x_values, heuristic_y_values, color="#8d99ae", linewidth=2.3, label="Heuristic trend")
ax.axhline(0.0, color="#6c757d", linewidth=1.0, alpha=0.75)
ax.axvline(0.0, color="#6c757d", linewidth=1.0, alpha=0.30)
ax.set_xlabel("Change in stance")
ax.set_ylabel("Change in log word rate")
ax.legend(frameon=False, loc="best", fontsize=LEGEND_SIZE)
ax.grid(alpha=0.16, linewidth=0.6)
fig.tight_layout()
save_figure(fig, EXP2_DIR / "exp2_heuristic_relationship.pdf")
if SAVE_PLOTS:
    fig.savefig(EXP2_DIR / "exp2_heuristic_relationship.png", format="png", dpi=300, bbox_inches="tight", pad_inches=0.30)
plt.show()


## Exp 4: Implicature Flow

In [ ]:
exp4_report = read_json(EXP4_DIR / "implicature_flow_global_report.json")
episode_df = read_exp4_episode_metrics(exp4_report, EXP4_DIR / "exp4_episode_metrics.csv")
lag_df = read_exp4_plot_data(EXP4_DIR / "exp4_lag_samples.csv", "Exp4 lag samples")
flow_df = read_exp4_plot_data(EXP4_DIR / "exp4_flow_edges.csv", "Exp4 flow edges")
require_columns(episode_df, ["episode_id", "turn_count", "total_assumptions", "accommodated", "dark_matter", "conversion_rate", "mean_lag"], "Exp4 episode metrics")
require_columns(lag_df, ["episode_id", "lag_seconds"], "Exp4 lag samples")
require_columns(flow_df, ["episode_id", "source_turn", "target_turn", "count", "mean_lag_seconds"], "Exp4 flow edges")

global_metrics = exp4_report["global_metrics"]
turn_counts = pd.to_numeric(episode_df["turn_count"], errors="coerce")
conversion_episode_df = episode_df[turn_counts >= 10].copy()
if conversion_episode_df.empty:
    raise ValueError("Exp4 conversion-rate plot has no episodes with at least 10 turns.")
filtered_total_assumptions = float(pd.to_numeric(conversion_episode_df["total_assumptions"]).sum())
if filtered_total_assumptions <= 0.0:
    raise ValueError("Exp4 conversion-rate plot has no assumptions after filtering episodes with fewer than 10 turns.")
global_conv = 100.0 * float(pd.to_numeric(conversion_episode_df["accommodated"]).sum()) / filtered_total_assumptions
lag_stats = global_metrics["lag_distribution_seconds"]

fig, ax = plt.subplots(figsize=(11.0, 6.5))
sns.histplot(pd.to_numeric(conversion_episode_df["conversion_rate"]), bins=25, color="#4C78A8", edgecolor="white", ax=ax)
ax.axvline(global_conv, color="#E45756", linestyle="--", linewidth=2, label=f"Filtered avg = {global_conv:.1f}%")
ax.set_xlabel("Conversion Rate (%)")
ax.set_ylabel("Episode Count")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, EXP4_DIR / "conversion_rate_distribution.pdf")
plt.show()

lag_values = finite_numeric(lag_df["lag_seconds"])
positive_lag_values = lag_values[lag_values > 0.0]
if len(positive_lag_values) == 0:
    raise ValueError("Exp4 lag samples have no positive lag_seconds values for log-scale plotting.")
log_bins = np.geomspace(float(np.min(positive_lag_values)), float(np.max(positive_lag_values)), 40)
fig, ax = plt.subplots(figsize=(11.0, 6.5))
sns.histplot(positive_lag_values, bins=log_bins, color="#72B7B2", edgecolor="white", ax=ax)
ax.set_xscale("log")
ax.axvline(float(lag_stats["median"]), color="#DD8452", linestyle="--", linewidth=2, label=f"Median = {float(lag_stats['median']):.1f}s")
ax.set_xlabel("Lag (seconds, log scale)")
ax.set_ylabel("Count")
ax.legend(frameon=False)
fig.tight_layout()
save_figure(fig, EXP4_DIR / "lag_distribution.pdf")
plt.show()

scatter_df = episode_df.copy()
scatter_df["dark_ratio"] = 100.0 * scatter_df["dark_matter"] / scatter_df["total_assumptions"].clip(lower=1)
fig, ax = plt.subplots(figsize=(10.5, 7.5))
sns.scatterplot(data=scatter_df, x="conversion_rate", y="mean_lag", size="total_assumptions", hue="dark_ratio", palette="viridis", sizes=(20, 180), alpha=0.75, ax=ax)
ax.set_xlabel("Conversion Rate (%)")
ax.set_ylabel("Mean Time-to-Surface (s)")
fig.tight_layout()
save_figure(fig, EXP4_DIR / "episode_accommodation_profile.pdf")
plt.show()

heat = flow_df.groupby(["source_turn", "target_turn"], as_index=False)["count"].sum()
if heat.empty:
    raise ValueError("Exp4 flow edges are empty; cannot render heatmap.")
heat = heat[(heat["source_turn"] < 50) & (heat["target_turn"] < 50)].copy()
if heat.empty:
    raise ValueError("Exp4 heatmap has no flow edges with both turns less than 50.")
pivot = heat.pivot(index="source_turn", columns="target_turn", values="count").fillna(0)
if pivot.empty:
    raise ValueError("Exp4 heatmap pivot is empty after caps.")
fig, ax = plt.subplots(figsize=(10.5, 8.5))
cmap = sns.color_palette("mako", as_cmap=True)
cmap.set_under("white")
sns.heatmap(np.log1p(pivot), cmap=cmap, vmin=1e-9, ax=ax, cbar_kws={"label": "log(1 + accommodation count)"})
ax.set_xlabel("Explicit Claim Turn")
ax.set_ylabel("Assumption Turn")
fig.tight_layout()
save_figure(fig, EXP4_DIR / "turn_lag_heatmap.pdf")
plt.show()

fig, ax = plt.subplots(figsize=(11.0, 5.8))
total = int(global_metrics["total_assumptions"])
accommodated = int(global_metrics["total_accommodated"])
dark = int(global_metrics["dark_matter_count"])
ax.add_patch(plt.Rectangle((0.08, 0.35), 0.22, 0.3, color="#4C78A8", alpha=0.35))
ax.add_patch(plt.Rectangle((0.70, 0.58), 0.22, 0.2, color="#59A14F", alpha=0.45))
ax.add_patch(plt.Rectangle((0.70, 0.12), 0.22, 0.2, color="#E15759", alpha=0.45))
ax.text(0.19, 0.5, f"Implicit Pool\n{total:,}", ha="center", va="center", fontsize=ANNOTATION_SIZE + 2, weight="bold")
ax.text(0.81, 0.68, f"Explicit Record\n{accommodated:,}", ha="center", va="center", fontsize=ANNOTATION_SIZE, weight="bold")
ax.text(0.81, 0.22, f"Dark Matter\n{dark:,}", ha="center", va="center", fontsize=ANNOTATION_SIZE, weight="bold")
for y0, y1, color in [(0.52, 0.68, "#59A14F"), (0.46, 0.22, "#E15759")]:
    path_x = np.linspace(0.30, 0.70, 80)
    t = np.linspace(0, 1, 80)
    ease = 3 * t**2 - 2 * t**3
    ax.plot(path_x, y0 + (y1 - y0) * ease, color=color, linewidth=10, alpha=0.4, solid_capstyle="round")
ax.text(0.50, 0.67, f"{float(global_metrics['conversion_rate_percent']):.1f}%", ha="center", va="center", fontsize=ANNOTATION_SIZE, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "none", "alpha": 0.85})
ax.text(0.50, 0.24, f"{float(global_metrics['dark_matter_ratio_percent']):.1f}%", ha="center", va="center", fontsize=ANNOTATION_SIZE, bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "none", "alpha": 0.85})
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
fig.tight_layout()
save_figure(fig, EXP4_DIR / "global_implicature_flow_summary.pdf")
plt.show()


In [ ]:
EXP4_SANKEY_LIMIT = 50
if flow_df.empty:
    raise ValueError("Exp4 flow edges are empty; cannot render Sankey diagrams.")

for episode_id in sorted(flow_df["episode_id"].astype(str).unique())[:EXP4_SANKEY_LIMIT]:
    episode_edges = flow_df[flow_df["episode_id"].astype(str) == episode_id].copy()
    turns = sorted(set(episode_edges["source_turn"].astype(int)) | set(episode_edges["target_turn"].astype(int)))
    if not turns:
        raise ValueError(f"Exp4 episode has no turn nodes for Sankey: {episode_id}")
    n_turns = len(turns)
    y_positions = [0.5] if n_turns == 1 else list(np.linspace(0.95, 0.05, n_turns))
    palette = sns.color_palette("husl", n_turns)
    turn_colors = {turn: f"rgba({int(r * 255)},{int(g * 255)},{int(b * 255)},0.65)" for turn, (r, g, b) in zip(turns, palette)}
    labels = [f"A{turn}" for turn in turns] + [f"C{turn}" for turn in turns]
    lookup = {("A", turn): index for index, turn in enumerate(turns)}
    lookup.update({("C", turn): index + n_turns for index, turn in enumerate(turns)})
    sources = [lookup[("A", int(row.source_turn))] for row in episode_edges.itertuples(index=False)]
    targets = [lookup[("C", int(row.target_turn))] for row in episode_edges.itertuples(index=False)]
    values = episode_edges["count"].astype(int).tolist()
    link_colors = [turn_colors[int(row.source_turn)] for row in episode_edges.itertuples(index=False)]
    customdata = episode_edges[["mean_lag_seconds", "count"]].astype(float).to_numpy()
    fig = go.Figure(go.Sankey(arrangement="fixed", node={"pad": 10, "thickness": 12, "line": {"color": "rgba(80,80,80,0.35)", "width": 0.5}, "label": labels, "color": [turn_colors[turn] for turn in turns] + ["rgba(170,170,170,0.85)"] * n_turns, "x": [0.08] * n_turns + [0.92] * n_turns, "y": y_positions + y_positions, "hovertemplate": "%{label}<extra></extra>"}, link={"source": sources, "target": targets, "value": values, "color": link_colors, "customdata": customdata, "hovertemplate": "From %{source.label} to %{target.label}<br>Count: %{value}<br>Mean lag: %{customdata[0]:.1f}s<extra></extra>"}))
    fig.update_layout(font={"size": TICK_SIZE}, width=1100, height=max(550, int(26 * n_turns + 180)), margin={"l": 40, "r": 40, "t": 30, "b": 30})
    if SAVE_PLOTS:
        EXP4_SANKEY_DIR.mkdir(parents=True, exist_ok=True)
        save_plotly_pdf(fig, EXP4_SANKEY_DIR / f"sankey_{episode_id}.pdf")
    fig.show()


## Exp 5: Processing Load

In [ ]:
for exp5_dir in EXP5_RESULT_DIRS:
    turn_df = read_csv(exp5_dir / "exp5_turn_level_features.csv")
    curve_df = read_csv(exp5_dir / "exp5_probability_curves.csv")
    counts_df = read_csv(exp5_dir / "exp5_response_type_counts.csv")
    require_columns(turn_df, ["assumption_count_in_turn", "gap_to_next_sec", "implicature_load", "next_response_type"], "Exp5 turn features")
    require_columns(curve_df, ["implicature_load"], "Exp5 probability curves")
    require_columns(counts_df, ["response_type", "count"], "Exp5 response counts")

    probability_columns = sorted(
        column
        for column in curve_df.columns
        if column.startswith("p_") and not column.endswith(("_ci95_low", "_ci95_high"))
    )
    interval_columns = [column for probability_column in probability_columns for column in [f"{probability_column}_ci95_low", f"{probability_column}_ci95_high"]]
    require_columns(curve_df, interval_columns, "Exp5 probability curve intervals")

    fig, ax = plt.subplots(figsize=(11.8, 6.0))
    x = curve_df["implicature_load"].astype(float).to_numpy()
    xmin, xmax = robust_xlim(x, 0.01, 0.99, None)
    for column in probability_columns:
        y_values = curve_df[column].astype(float).to_numpy()
        low_values = curve_df[f"{column}_ci95_low"].astype(float).to_numpy()
        high_values = curve_df[f"{column}_ci95_high"].astype(float).to_numpy()
        label = column.replace("p_", "")
        line = ax.plot(x, y_values, linewidth=2.0, label=label)[0]
        ax.fill_between(x, low_values, high_values, color=line.get_color(), alpha=0.14, linewidth=0.0)
    ax.set_xlabel("Implicature Load L")
    ax.set_ylabel("Predicted probability")
    ax.set_ylim(0, 0.8)
    ax.set_xlim(xmin, xmax)
    ax.legend(frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=1, fontsize=LEGEND_SIZE)
    fig.subplots_adjust(left=0.12, right=0.74, bottom=0.16, top=0.94)
    save_figure(fig, exp5_dir / "exp5_probability_curves.pdf")
    plt.show()

    for x_column, xlabel, output_name in [
        ("assumption_count_in_turn", "Assumption count in turn", "exp5_assumption_count_vs_response_time.pdf"),
        ("implicature_load", "Implicature Load L", "exp5_implicature_load_vs_response_time.pdf"),
    ]:
        pairs = turn_df[[x_column, "gap_to_next_sec"]].apply(pd.to_numeric, errors="coerce").dropna()
        pairs = pairs[pairs["gap_to_next_sec"] >= 0]
        if len(pairs) < 3:
            raise ValueError(f"Exp5 scatter has fewer than 3 finite rows for {x_column}.")
        x_values = pairs[x_column].to_numpy(dtype=float)
        y_values = pairs["gap_to_next_sec"].to_numpy(dtype=float)
        xmin, xmax = robust_xlim(x_values, 0.01, 0.99, None)
        ymin, ymax = robust_xlim(y_values, 0.0, 0.995, 2000.0)
        mask = np.isfinite(x_values) & np.isfinite(y_values) & (x_values >= xmin) & (x_values <= xmax) & (y_values >= ymin) & (y_values <= ymax)
        x_values = x_values[mask]
        y_values = y_values[mask]
        if len(x_values) < 3:
            raise ValueError(f"Exp5 scatter has fewer than 3 visible rows for {x_column}.")
        rng = np.random.default_rng(42)
        if len(x_values) > 30000:
            keep = rng.choice(len(x_values), size=30000, replace=False)
            x_plot = x_values[keep]
            y_plot = y_values[keep]
        else:
            x_plot = x_values
            y_plot = y_values
        fig, ax = plt.subplots(figsize=(10.0, 6.0))
        sns.scatterplot(x=x_plot, y=y_plot, s=10, alpha=0.18, linewidth=0, color="#4C78A8", ax=ax)
        sns.regplot(x=x_values, y=y_values, scatter=False, ci=None, line_kws={"color": "#E45756", "linewidth": 2.2}, ax=ax)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Response latency (seconds)")
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        fig.tight_layout()
        save_figure(fig, exp5_dir / output_name)
        plt.show()

    response_order = counts_df["response_type"].astype(str).tolist()
    by_class = {}
    for response_type in response_order:
        values = pd.to_numeric(turn_df.loc[turn_df["next_response_type"].astype(str) == response_type, "implicature_load"], errors="coerce").dropna().astype(float).to_numpy()
        values = values[np.isfinite(values)]
        if len(values) >= 5:
            by_class[response_type] = values
    if not by_class:
        raise ValueError("Exp5 ridge plot has no response classes with at least 5 finite load values.")
    all_values = np.concatenate(list(by_class.values()))
    xmin, xmax = robust_xlim(np.log1p(all_values), 0.0, 0.995, None)
    fig, axes = plt.subplots(len(by_class), 1, figsize=(11.0, 1.8 * len(by_class)), sharex=True)
    axes = [axes] if len(by_class) == 1 else list(axes)
    colors = sns.color_palette("Set2", len(by_class))
    for axis, (response_type, values), color in zip(axes, by_class.items(), colors):
        plot_values = np.log1p(values)
        visible = plot_values[(plot_values >= xmin) & (plot_values <= xmax)]
        if len(visible) < 5:
            visible = plot_values
        sns.kdeplot(x=visible, fill=True, alpha=0.7, linewidth=1.0, color=color, cut=0, clip=(xmin, xmax), ax=axis)
        sns.kdeplot(x=visible, fill=False, linewidth=1.0, color="black", cut=0, clip=(xmin, xmax), ax=axis)
        axis.set_ylabel(response_type, rotation=0, ha="right", va="center", labelpad=35)
        axis.set_yticks([])
        axis.grid(axis="x", alpha=0.2)
        for spine in ("top", "right", "left"):
            axis.spines[spine].set_visible(False)
        axis.set_xlim(xmin, xmax)
    raw_ticks = np.asarray([0, 5, 10, 20, 40, 80, 160, 320, 640, 1280], dtype=float)
    ticks = np.log1p(raw_ticks)
    visible_ticks = ticks[(ticks >= xmin) & (ticks <= xmax)]
    axes[-1].set_xticks(visible_ticks)
    axes[-1].set_xticklabels([str(int(round(np.expm1(value)))) for value in visible_ticks])
    axes[-1].set_xlabel("Implicature Load L")
    fig.subplots_adjust(bottom=0.12, top=0.98)
    save_figure(fig, exp5_dir / "exp5_load_ridge_by_response_type.pdf")
    plt.show()


## Exp 6: Quantity Repair Cascades

In [ ]:
trigger_df = read_csv(EXP6_DIR / "exp6_trigger_outcomes.csv")
event_df = read_csv(EXP6_DIR / "exp6_cascade_events.csv")
hawkes_df = read_csv(EXP6_DIR / "exp6_hawkes_summary.csv")
require_columns(trigger_df, ["trigger_type", "cascade_duration_sec", "repair_event_count", "challenge_event_count", "continuation_event_count", "topic_shift_indicator", "next_turn_latency_sec", "episode_id", "trigger_turn_idx", "observation_horizon_sec"], "Exp6 trigger outcomes")
require_columns(event_df, ["trigger_type", "episode_id", "trigger_turn_idx", "event_type", "relative_start_sec"], "Exp6 cascade events")
require_columns(hawkes_df, ["trigger_type", "branching_ratio", "alpha"], "Exp6 Hawkes summary")

def exp6_metric_mean(column_name: str, source: str, trigger_type: str) -> float:
    if source == "hawkes":
        rows = hawkes_df[hawkes_df["trigger_type"] == trigger_type]
        return float(rows[column_name].iloc[0]) if not rows.empty and pd.notna(rows[column_name].iloc[0]) else 0.0
    group = trigger_df[trigger_df["trigger_type"] == trigger_type]
    if column_name == "hawkes_event_count":
        values = group["repair_event_count"].astype(float).to_numpy() + group["challenge_event_count"].astype(float).to_numpy()
    else:
        values = group[column_name].astype(float).to_numpy()
    values = values[np.isfinite(values)]
    return float(np.mean(values)) if len(values) else 0.0

def format_exp6_metric_value(value: float) -> str:
    if abs(value) >= 10.0:
        return f"{value:.1f}"
    if abs(value) >= 1.0:
        return f"{value:.2f}"
    return f"{value:.3f}"

overview_metric_specs = [
    ("branching_ratio", "Excitation alpha / beta", "hawkes"),
    ("alpha", "Excitation alpha", "hawkes"),
    ("cascade_duration_sec", "Cascade duration (s)", "trigger"),
    ("hawkes_event_count", "Repair/challenge events", "trigger"),
    ("topic_shift_indicator", "Topic-shift rate", "trigger"),
    ("next_turn_latency_sec", "Next-turn latency (s)", "trigger"),
]
raw_means = np.zeros((len(EXP6_TRIGGER_ORDER), len(overview_metric_specs)), dtype=float)
normalized_means = np.zeros_like(raw_means)
for metric_index, (column_name, axis_title, source) in enumerate(overview_metric_specs):
    metric_means = [exp6_metric_mean(column_name, source, trigger_type) for trigger_type in EXP6_TRIGGER_ORDER]
    max_mean = max(metric_means)
    if max_mean <= 0.0:
        raise ValueError(f"Cannot normalize Exp6 overview metric with no positive values: {axis_title}")
    for trigger_index, mean in enumerate(metric_means):
        raw_means[trigger_index, metric_index] = mean
        normalized_means[trigger_index, metric_index] = mean / max_mean
y_positions = np.arange(len(overview_metric_specs), dtype=float)
bar_height = 0.34
labels = [EXP6_TRIGGER_LABELS[trigger_type] for trigger_type in EXP6_TRIGGER_ORDER]
colors = [EXP6_TRIGGER_COLORS[trigger_type] for trigger_type in EXP6_TRIGGER_ORDER]
fig, (bar_axis, text_axis) = plt.subplots(1, 2, figsize=(13.8, 7.4), gridspec_kw={"width_ratios": [4.8, 1.45], "wspace": 0.08})
for trigger_index, trigger_type in enumerate(EXP6_TRIGGER_ORDER):
    offset = (trigger_index - (len(EXP6_TRIGGER_ORDER) - 1) / 2.0) * bar_height
    bar_axis.barh(y_positions + offset, normalized_means[trigger_index], height=bar_height, color=colors[trigger_index], edgecolor="#1f2933", linewidth=0.6, label=labels[trigger_index])
raw_value_labels = [
    f"{format_exp6_metric_value(raw_means[0, metric_index])} / {format_exp6_metric_value(raw_means[1, metric_index])}"
    for metric_index in range(len(overview_metric_specs))
]
bar_axis.set_yticks(y_positions)
bar_axis.set_yticklabels([axis_title for _, axis_title, _ in overview_metric_specs], fontsize=TICK_SIZE)
bar_axis.set_xlim(0.0, 1.05)
bar_axis.set_xlabel("Normalized mean within each metric (larger trigger type = 1.0)", fontsize=LABEL_SIZE)
bar_axis.grid(axis="x", alpha=0.18, linewidth=0.7)
bar_axis.invert_yaxis()
text_axis.set_xlim(0.0, 1.0)
text_axis.set_ylim(bar_axis.get_ylim())
text_axis.set_xticks([])
text_axis.set_yticks([])
for spine in text_axis.spines.values():
    spine.set_visible(False)
text_axis.text(0.0, 1.025, "Raw mean", transform=text_axis.transAxes, ha="left", va="bottom", fontsize=ANNOTATION_SIZE - 2, weight="bold")
text_axis.text(0.0, 0.975, "Under / Over", transform=text_axis.transAxes, ha="left", va="bottom", fontsize=ANNOTATION_SIZE - 2, weight="bold")
for y_position, raw_value_label in zip(y_positions, raw_value_labels, strict=True):
    text_axis.text(0.0, y_position, raw_value_label, ha="left", va="center", fontsize=ANNOTATION_SIZE - 2)
handles, legend_labels = bar_axis.get_legend_handles_labels()
fig.legend(handles, legend_labels, frameon=False, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, -0.005), fontsize=LEGEND_SIZE)
fig.subplots_adjust(left=0.22, right=0.98, top=0.88, bottom=0.18)
save_figure(fig, EXP6_DIR / "exp6_cascade_overview.pdf")
plt.show()

def sample_exp6_triggers(trigger_data: pd.DataFrame, trigger_type: str, sample_count: int) -> pd.DataFrame:
    group = trigger_data[trigger_data["trigger_type"] == trigger_type].copy()
    if group.empty:
        return group
    group["hawkes_event_count"] = group["repair_event_count"].astype(int) + group["challenge_event_count"].astype(int)
    ordered = group.sort_values(["hawkes_event_count", "cascade_duration_sec", "continuation_event_count", "episode_id", "trigger_turn_idx"], ascending=[False, False, False, True, True]).reset_index(drop=True)
    if len(ordered) <= sample_count:
        return ordered
    top_count = max(1, sample_count // 2)
    remaining = ordered.iloc[top_count:].copy()
    rng = np.random.default_rng(42)
    selected = rng.choice(len(remaining), size=sample_count - top_count, replace=False)
    return pd.concat([ordered.head(top_count), remaining.iloc[np.sort(selected)]], ignore_index=True)

fig, axes = plt.subplots(len(EXP6_TRIGGER_ORDER), 1, figsize=(15.0, 10.8), sharex=True)
axes = np.atleast_1d(axes).tolist()
for axis, trigger_type in zip(axes, EXP6_TRIGGER_ORDER, strict=True):
    sampled = sample_exp6_triggers(trigger_df, trigger_type, EXP6_SAMPLE_CASCADE_COUNT)
    axis.axvline(0.0, color="#495057", linewidth=1.3, linestyle="--", alpha=0.8)
    axis.set_ylabel("Cascade sample", fontsize=LABEL_SIZE)
    axis.grid(axis="x", alpha=0.14, linewidth=0.8)
    axis.grid(axis="y", alpha=0.08, linewidth=0.6)
    if sampled.empty:
        raise ValueError(f"No Exp6 sampled triggers for {trigger_type}.")
    for row_position, trigger_row in enumerate(sampled.itertuples(index=False)):
        axis.hlines(y=row_position, xmin=0.0, xmax=float(trigger_row.observation_horizon_sec), color="#d9e2ec", linewidth=1.7, alpha=0.95)
        mask = (event_df["trigger_type"] == trigger_type) & (event_df["episode_id"] == trigger_row.episode_id) & (event_df["trigger_turn_idx"] == trigger_row.trigger_turn_idx)
        for event_type, group in event_df[mask].groupby("event_type", observed=False):
            axis.scatter(group["relative_start_sec"].astype(float).to_numpy(), np.full(len(group), row_position, dtype=float), color=EXP6_EVENT_COLORS[event_type], marker=EXP6_EVENT_MARKERS[event_type], s=74, alpha=0.94, zorder=4)
    axis.set_ylim(-0.75, len(sampled) - 0.25)
    axis.set_yticks(np.arange(len(sampled), dtype=float))
    axis.set_yticklabels([str(index + 1) for index in range(len(sampled))], fontsize=TICK_SIZE)
    axis.set_xlim(0.0, 180.0)
axes[-1].set_xlabel("Seconds from trigger turn", fontsize=LABEL_SIZE)
handles = [plt.Line2D([0], [0], marker=EXP6_EVENT_MARKERS[event_type], color="none", markerfacecolor=EXP6_EVENT_COLORS[event_type], markeredgecolor=EXP6_EVENT_COLORS[event_type], markersize=12) for event_type in EXP6_EVENT_MARKERS]
labels = [event_type.replace("_", " ").title() for event_type in EXP6_EVENT_MARKERS]
fig.legend(handles, labels, loc="center right", ncol=1, frameon=False, bbox_to_anchor=(0.985, 0.5), fontsize=LABEL_SIZE, labelspacing=1.0, handletextpad=0.5)
fig.subplots_adjust(top=0.95, left=0.10, right=0.70, bottom=0.12, hspace=0.34)
save_figure(fig, EXP6_DIR / "exp6_event_cascades.pdf")
plt.show()


## Exp 7: Social Power

In [ ]:
curve_df = read_csv(EXP7_DIR / "exp7_interaction_curve.csv")
require_columns(
    curve_df,
    [
        "speaker_role",
        "violation_type",
        "severity",
        "predicted_policed_probability",
        "predicted_policed_probability_ci95_low",
        "predicted_policed_probability_ci95_high",
    ],
    "Exp7 interaction curve",
)
colors = {"guest": "#b42318", "host": "#175cd3"}
severity_to_label = {EXP7_VIOLATION_SEVERITY[violation]: violation for violation in EXP7_VIOLATION_ORDER}
x_values = [EXP7_VIOLATION_SEVERITY[violation] for violation in EXP7_VIOLATION_ORDER]
fig, ax = plt.subplots(figsize=(10.2, 6.1))
ax.set_facecolor("#ffffff")
ax.grid(axis="y", color="#e5e7eb", linewidth=1.0)
ax.set_axisbelow(True)
for role in EXP7_ROLE_ORDER:
    role_rows = curve_df[curve_df["speaker_role"] == role].sort_values("severity")
    if role_rows.empty:
        raise ValueError(f"Exp7 interaction curve is missing role: {role}")
    x_role_values = role_rows["severity"].astype(int).to_numpy()
    y_values = role_rows["predicted_policed_probability"].astype(float).to_numpy()
    low_values = role_rows["predicted_policed_probability_ci95_low"].astype(float).to_numpy()
    high_values = role_rows["predicted_policed_probability_ci95_high"].astype(float).to_numpy()
    ax.plot(x_role_values, y_values, color=colors[role], linewidth=3.0, marker="o", markersize=7, label=role.title())
    ax.errorbar(x_role_values, y_values, yerr=np.vstack([y_values - low_values, high_values - y_values]), fmt="none", ecolor=colors[role], elinewidth=1.5, capsize=4, alpha=0.72)
ax.set_xlabel("Violation type", fontsize=LABEL_SIZE)
ax.set_ylabel("Predicted policing probability", fontsize=LABEL_SIZE)
ax.set_xticks(x_values)
ax.set_xticklabels([severity_to_label[x_value] for x_value in x_values], fontsize=TICK_SIZE)
probabilities = curve_df["predicted_policed_probability_ci95_high"].astype(float).to_numpy()
max_probability = float(np.nanmax(probabilities)) if len(probabilities) else 0.4
ax.set_ylim(0.0, min(1.0, max(0.45, math.ceil((max_probability + 0.03) * 20.0) / 20.0)))
ax.legend(frameon=False, loc="upper right", fontsize=LEGEND_SIZE)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
save_figure(fig, EXP7_DIR / "exp7_status_shield_plot.pdf")
plt.show()
